# Merging NOAA, egalei, and ERA5

The goal of this notebook is to merge the NOAA (as a time series) and eaglei data using FIPS, datetime, and latitude/longitude (of the county centroid) as a multi-index

Then it will merge this with corresponding ERA5 data

## Imports

In [ ]:
import pandas as pd
import dask.dataframe as dd
import fastparquet

# Indexing NOAA and eaglei by time

The eaglei data wasn't saved with a timeseries index, and didn't have FIPS as part of its index
This code will covert the original exported eaglei data into the right type of time series

In [ ]:
#Load the Data/eaglei_data/eaglei_outages_with_county_info.parquet file
eaglei = pd.read_parquet('Data/Merged_Data/eaglei_outages_with_county_info.parquet')

#Make datetime and FIPS a multiindex for eaglei
eaglei['time'] = pd.to_datetime(eaglei['datetime'])
eaglei.set_index(['time', 'FIPS'], inplace=True)

#Export eaglei as a parquet file
eaglei.to_parquet('Data/Merged_Data/eaglei_outages_with_county_info_timeseries.parquet')

# Merging NOAA and EAGLEI

The datasets we want to work with are pretty large, so we'll load them as dask dataframes

In [ ]:
#Load the Data/eaglei_data/eaglei_outages_with_county_info.parquet file
eaglei = dd.read_parquet('Data/Merged_Data/eaglei_outages_with_county_info_timeseries.parquet')

#Load the Data/NOAA_Cleaned_ExplodedFIPS.parquet file
noaa = dd.read_parquet('Data/NOAA_Cleaned_Data/NOAA_Timeseries.parquet')

#Merge eaglei and noaa based on their indices
eaglei_noaa = eaglei.merge(noaa, left_index=True, right_index=True, how='left')

# Adding Geospatial Indices to eaglei-NOAA

To merge with ERA5 data, the eaglei-NOAA data will need to have latitude and longitude as indices/coordinates.

In [ ]:
#Use fips_code, datetime, centroid_latitude and centroid_longitude as indices.
#Since dask doesn't support multiindexes, we'll keep these as variables as well
eaglei_noaa.set_index(['fips_code', 'datetime', 'centroid_latitude', 'centroid_longitude'], inplace=True, drop=False)

#Rename the indices FIPS, time, latitude, and longitude
eaglei_noaa.index.names = ['FIPS', 'time', 'latitude', 'longitude']

#Export eaglei_noaa as a parquet file eaglei_noaa_latlon
eaglei_noaa.to_parquet('Data/Merged_Data/eaglei_noaa_latlon.parquet')

# Merging NOAA-eaglei and ERA5

We can use pyarrow or dask to merge the parquet files

In [ ]:
parquet_files = [
    'Data/ERA5_Data/ERA5_2014.parquet',
    'Data/ERA5_Data/ERA5_2015.parquet',
    'Data/ERA5_Data/ERA5_2016.parquet',
    'Data/ERA5_Data/ERA5_2017.parquet',
    'Data/ERA5_Data/ERA5_2018.parquet',
    'Data/ERA5_Data/ERA5_2019.parquet',
    'Data/ERA5_Data/ERA5_2020.parquet',
    'Data/ERA5_Data/ERA5_2021.parquet',
    'Data/ERA5_Data/ERA5_2022.parquet',
    'Data/ERA5_Data/ERA5_2023.parquet'
]

for file in parquet_files:
    # Load the ERA data and merge it with the already-(partially)-eaglei_noaa data
    era = pd.read_parquet(file)
    eaglei_noaa = eaglei_noaa.merge(era, how="left", on=['time', 'latitude', 'longitude'])

    #In eaglei_noaa, for each pair of identical columns ending with _x and _y, combine them into a new column, taking the non-NaN value if it exists
    #This can be done by iterating over the columns and checking for pairs that end with _x and _y
    
    # Create a list to hold the new column names
    new_columns = []
    for col in eaglei_noaa.columns:
    # Check if the column ends with _x
        if col.endswith('_x'):
            # Create the corresponding column name by removing _x and adding _new
            new_col = col[:-2]  # Remove the last two characters (_x)
            new_columns.append(new_col)
            # Create the new column by taking the non-NaN values from both columns
            eaglei_noaa[new_col] = eaglei_noaa[col].combine_first(eaglei_noaa[col[:-2] + '_y'])
            # Drop the original columns to avoid duplicates
            eaglei_noaa.drop(columns=[col, col[:-2] + '_y'], inplace=True)
    # Remove the era data to save memory
    del era

#Save merged as a parquet file using fastparquet partitioning on fips_code
eaglei_noaa.to_parquet(
    'Data/Merged_Data/eaglei_noaa_era5.parquet',
    engine='fastparquet',
    partition_cols=['fips_code']
)

#Save merged as a parquet file
eaglei_noaa.to_parquet(
    'Data/Merged_Data/eaglei_noaa_era5_full.parquet',
    engine='fastparquet'
)